**PRÁCTICA 5 · PASO 0: PREPARACIÓN DEL DATASET DE EMOCIONES**

El modelo biométrico propio clasifica **7 expresiones**: las 6 de FER-2013 que conservamos (`angry`, `fear`, `happy`, `neutral`, `sad`, `surprise`) más una clase nueva, **`duckface`** (morritos), que no existe en FER y que hemos construido nosotros.

| Origen | Clases | Formato |
|---|---|---|
| FER-2013 (Kaggle) | angry, fear, happy, neutral, sad, surprise | 48x48, gris, cara ya recortada |
| Selfie-dataset (columna `duck_face`) | duckface | fotos de móvil en color → recortadas con el detector DNN |
| ~~Roboflow `middle-tongue-duckface`~~ | duckface | **descartado** (ver sección 3) |

Decisiones sobre el dataset:
- **`disgust` eliminado**: en FER-2013 solo tiene 436 imágenes de train (1,4 %), en las primeras pruebas su recall se quedó en 0,32–0,46 y su mezcla con angry empeoraba ambas clases.
- **Formato común**: las caras nuevas se convierten al formato FER (gris, 48x48) para que todas las clases lleguen igual al modelo.
- **Mismo recorte que en la webcam**: el recorte se hace con `comun.recortar_cara`, la misma función que usan los prototipos en directo, para que el encuadre de entrenamiento e inferencia sea idéntico.

In [ ]:
import os, random, shutil
from pathlib import Path

import cv2
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from comun import DetectorCaras, recortar_cara, EXTENSIONES
from comun import EMOCIONES

detector = DetectorCaras(conf_min=0.5)

**1. DUCKFACE A PARTIR DEL SELFIE-DATASET**

`Selfie-dataset.tar.gz` contiene 46.836 selfies con 36 atributos binarios por imagen; la columna 23 es `duck_face`. Para cada selfie positivo:
1. Se detecta la cara con el DNN (ResNet10-SSD) y se toma la más confiada.
2. Se recorta con `recortar_cara` (5 % hacia dentro, igual que la webcam).
3. Se pasa a gris y se reduce a 48x48 con `INTER_AREA`.
4. Se reparte 80 % train / 20 % test con semilla fija.

Las imágenes en las que el detector no encuentra cara se descartan (suelen ser selfies de cuerpo entero, de espaldas o muy oscuros).

> Requiere descomprimir antes `dataset/Selfie-dataset.tar.gz` dentro de `dataset/`. Si ya existe `dataset/train/duckface`, pon `REGENERAR = True` para borrarla y volver a crearla.

In [ ]:
LABELS = "dataset/Selfie-dataset/Selfie-dataset/selfie_dataset.txt"
IMGS = "dataset/Selfie-dataset/Selfie-dataset/images"
COL_DUCK = 23           # columna duck_face
TEST = 0.2
SEED = 42
REGENERAR = False       # True = borra dataset/{train,test}/duckface y la vuelve a generar


def cara_selfie(ruta):
    img = cv2.imread(ruta)
    if img is None:
        return None
    caja = detector.detectar(img)
    if not caja:
        return None
    x, y, w, h, _ = max(caja, key=lambda c: c[4])       # la detección más confiada
    cara = recortar_cara(cv2.cvtColor(img, cv2.COLOR_BGR2GRAY), (x, y, w, h))
    return None if cara is None else cv2.resize(cara, (48, 48), interpolation=cv2.INTER_AREA)


if not Path(LABELS).exists():
    print("No está descomprimido el Selfie-dataset: se omite este paso.")
elif Path("dataset/train/duckface").exists() and not REGENERAR:
    print("dataset/train/duckface ya existe (REGENERAR = False): se omite este paso.")
else:
    df = pd.read_csv(LABELS, sep=r"\s+", header=None)
    nombres = [str(n) if str(n).lower().endswith((".jpg", ".jpeg", ".png")) else f"{n}.jpg"
               for n in df.loc[df[COL_DUCK] == 1, 0]]
    random.seed(SEED)
    random.shuffle(nombres)
    n_test = int(round(len(nombres) * TEST))
    for split, lista in {"test": nombres[:n_test], "train": nombres[n_test:]}.items():
        destino = Path("dataset", split, "duckface")
        if REGENERAR and destino.exists():
            shutil.rmtree(destino)
        destino.mkdir(parents=True, exist_ok=True)
        ok = descartadas = 0
        for nombre in lista:
            cara = cara_selfie(os.path.join(IMGS, nombre))
            if cara is None:
                descartadas += 1
                continue
            cv2.imwrite(str(destino / (Path(nombre).stem + ".png")), cara)
            ok += 1
        print(f"{split}/duckface: {ok} guardadas | {descartadas} sin cara o ilegibles")

**2. RECUENTO FINAL POR CLASE**

Se comprueba que las carpetas coinciden con `comun.EMOCIONES` y cuántas imágenes tiene cada clase. El desbalance (happy ≈ 24 % frente a duckface ≈ 6 %) se compensa después en el entrenamiento con `class_weight`.

In [ ]:
filas = {}
for split in ("train", "test"):
    carpetas = sorted(p.name for p in Path("dataset", split).iterdir() if p.is_dir() and not p.name.startswith("_"))
    if carpetas != sorted(EMOCIONES):
        print(f"[aviso] dataset/{split} tiene {carpetas}, se esperaba {sorted(EMOCIONES)}")
    filas[split] = {e: sum(1 for f in Path("dataset", split, e).iterdir() if f.suffix.lower() in EXTENSIONES)
                    for e in EMOCIONES}
tabla = pd.DataFrame(filas)
tabla["train_%"] = (100 * tabla["train"] / tabla["train"].sum()).round(1)
display(tabla)

**3. ¿POR QUÉ SE DESCARTÓ ROBOFLOW?**

La primera versión añadía 199 caras de train y 8 de test del proyecto de Roboflow `middle-tongue-duckface`. Al revisarlas a mano vimos tres problemas:
1. **Ya eran recortes de cara con aumento de datos de Roboflow** (rotaciones de hasta ±45° con esquinas negras). Ninguna imagen de FER tiene esquinas negras, así que el modelo podía aprender "esquinas negras ⇒ duckface" (un *atajo*), no la expresión.
2. **Cada cara aparecía 3 veces** (las 3 variantes aumentadas que exporta Roboflow), lo que infla artificialmente la clase.
3. El script original tenía un fallo de coordenadas (mezclaba la caja del DNN en píxeles con coordenadas YOLO normalizadas), por lo que el recorte acababa siendo la imagen entera.

Por eso esas imágenes se apartaron a `dataset/_descartadas_roboflow/`. La celda siguiente muestra ejemplos de ambas fuentes para la memoria.

In [ ]:
def mosaico(rutas, n=16, titulo=""):
    rutas = sorted(rutas)[:n]
    if not rutas:
        print("Sin imágenes para:", titulo)
        return
    fig, ax = plt.subplots(1, len(rutas), figsize=(1.1 * len(rutas), 1.4))
    for a, r in zip(np.atleast_1d(ax), rutas):
        a.imshow(cv2.imread(str(r), 0), cmap="gray"); a.axis("off")
    fig.suptitle(titulo); plt.show()

mosaico(Path("dataset/train/duckface").glob("*.png"), titulo="Duckface (Selfie-dataset, se usan)")
mosaico(Path("dataset/_descartadas_roboflow").rglob("*.png"), titulo="Roboflow (descartadas)")
mosaico(Path("dataset/train/happy").glob("*"), titulo="FER-2013 · happy, como referencia")

**4. LIMITACIÓN CONOCIDA: SESGO DE DOMINIO EN DUCKFACE**

Todas las imágenes de `duckface` provienen de selfies de móvil (más resolución de origen, iluminación de interior, a veces filtros), mientras que el resto de clases provienen de FER-2013. Esto puede hacer que el modelo use pistas del *dominio* además de la expresión, y explica en parte que duckface tenga la precisión más alta (≈ 0,96 en la CNN). Para comprobarlo hay que probar en directo: poner morritos frente a la webcam y ver si se detecta, y comprobar que una cara neutra en un selfie no se clasifica como duckface.